# Lab EDA: GermanQuAD và GermanDPR
**Mục tiêu:** hiểu dữ liệu đủ sâu để mọi tham số cài đặt (độ dài, cách chia dev, định nghĩa "truy hồi đúng", cải tiến nào đáng làm) đều có căn cứ.

**Cách dùng notebook**
- Chạy lần lượt từ trên xuống. Runtime: CPU đủ cho phần thống kê; **chọn GPU T4** (Runtime > Change runtime type) trước khi chạy Lab D1 (encode dense).
- Mỗi lab có 4 phần: **Mục đích → Mã chạy → ✍️ Câu hỏi bạn tự trả lời → 🎯 Ô quyết định** (hàm `decide(...)`).
- Ô quyết định đã điền sẵn giá trị gợi ý. **Bạn phải đọc kết quả rồi sửa lại cho đúng với số đo của bạn** và viết lý do bằng lời của mình.
- Cuối notebook (Phần E) sẽ gom mọi quyết định thành bảng *Phát hiện → Quyết định → Phần cài đặt bị ảnh hưởng*.

**Thứ tự chạy:** A0 → A1 → A2 → B → C1 → C2 → C3 → C4 → C5 → C6 → C7 → C9 → D1 → C8 → D2 → D3 → D4 → E.
(C8 đo latency cần chỉ mục dense nên đặt sau D1.)

**Nguyên tắc:** mọi quyết định dựa trên **train/dev**. Không dùng test để chọn thiết kế.

## 0. Cài đặt môi trường

In [ ]:
!pip -q install "datasets<4" rank_bm25 sentence-transformers faiss-cpu nltk scikit-learn matplotlib pandas

In [ ]:
import os, re, json, random, time, hashlib, unicodedata
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

random.seed(42); np.random.seed(42)

# Lưu kết quả lên Drive (nếu mount được), nếu không thì lưu tạm trong /content
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/lab_eda_german"
except Exception:
    OUT = "/content/lab_eda_german"
for sub in ("tables", "figures"):
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)
print("Kết quả lưu tại:", OUT)

SUMMARY = {}          # số liệu đo được
DECISIONS = []        # các quyết định cài đặt

def save_table(df, name):
    df.to_csv(f"{OUT}/tables/{name}.csv", index=False)
    return df

def save_fig(name):
    plt.tight_layout(); plt.savefig(f"{OUT}/figures/{name}.png", dpi=130); plt.show()

def decide(lab, item, value, reason):
    """Ghi một quyết định cài đặt. Sửa value/reason theo số đo của bạn."""
    DECISIONS.append({"lab": lab, "quyet_dinh": item, "gia_tri": value, "ly_do": reason})
    print(f"[{lab}] {item} = {value}\n    lý do: {reason}")

## Phần A. Hiểu bộ dữ liệu (Tầng 1)
### Lab A0. Tải dữ liệu và đối chiếu số liệu bài báo
**Mục đích:** có dữ liệu thật, biết cấu trúc, kiểm số liệu bài báo (13.722 / 11.518 / 2.204 / 9.275 / 1.025).

Lưu ý: `datasets>=4` không còn hỗ trợ script tải dữ liệu, nên ô cài đặt đã ghim `datasets<4`. Nếu vẫn lỗi, thử nhánh parquet tự động ở dưới.

In [ ]:
from datasets import load_dataset

def load_any(name):
    for kw in ({}, {"trust_remote_code": True}, {"revision": "refs/convert/parquet"}):
        try:
            return load_dataset(name, **kw)
        except Exception as e:
            print(f"  {name} {kw} lỗi: {str(e)[:120]}")
    raise RuntimeError(f"Không tải được {name}")

qa  = load_any("deepset/germanquad")
dpr = load_any("deepset/germandpr")
print(qa); print(dpr)

In [ ]:
# Xem cấu trúc một mẫu của mỗi bộ. ĐỌC KỸ để biết tên trường thật.
print("=== GermanQuAD train[0] ===")
for k, v in qa["train"][0].items():
    print(f"{k:14s} {type(v).__name__:6s} {str(v)[:150]!r}")
print("\n=== GermanDPR train[0] ===")
for k, v in dpr["train"][0].items():
    print(f"{k:20s} {type(v).__name__:6s} {str(v)[:150]!r}")

In [ ]:
# --- Hàm đọc dữ liệu chịu được các biến thể cấu trúc (list-of-dict hoặc dict-of-list) ---
def norm(s):
    s = unicodedata.normalize("NFC", s).lower()
    return re.sub(r"\s+", " ", s).strip()

def pid(text):
    return hashlib.sha1(norm(text).encode("utf-8")).hexdigest()[:16]

def ctx_list(c):
    """Trả về list[str] các passage từ một trường ctxs."""
    if c is None: return []
    if isinstance(c, dict):
        t = c.get("text")
        return list(t) if isinstance(t, list) else ([t] if t else [])
    if isinstance(c, list):
        return [x["text"] if isinstance(x, dict) else str(x) for x in c]
    return [str(c)]

ex = dpr["train"][0]
K_POS  = [k for k in ex if "pos" in k.lower() and "neg" not in k.lower()][0]
K_HNEG = [k for k in ex if "hard" in k.lower()][0]
print("Trường positive:", K_POS, "| trường hard negative:", K_HNEG)

def qa_answers(x):
    a = x["answers"]
    return list(a["text"]) if isinstance(a, dict) else [y["text"] for y in a]

def qa_starts(x):
    a = x["answers"]
    return list(a["answer_start"]) if isinstance(a, dict) else [y["answer_start"] for y in a]

def dpr_records(split):
    recs = []
    for x in dpr[split]:
        pos = ctx_list(x[K_POS]); hn = ctx_list(x[K_HNEG])
        ans = x["answers"]; ans = [ans] if isinstance(ans, str) else list(ans)
        recs.append({"q": x["question"], "ans": ans, "pos": pos[0] if pos else None, "hneg": hn})
    return recs

DPR = {s: dpr_records(s) for s in dpr.keys()}
print({s: len(v) for s, v in DPR.items()})

In [ ]:
paper = {("germanquad","train"):11518, ("germanquad","test"):2204,
         ("germandpr","train"):9275,  ("germandpr","test"):1025}
rows = []
for (name, split), n_paper in paper.items():
    ds = qa if name == "germanquad" else dpr
    n = len(ds[split]) if split in ds else None
    rows.append({"bo": name, "split": split, "bai_bao": n_paper, "do_duoc": n,
                 "lech": None if n is None else n - n_paper})
a0 = save_table(pd.DataFrame(rows), "A0_so_lieu")
display(a0)
print("Tổng QuAD (train+test):", sum(len(qa[s]) for s in qa.keys()), "| bài báo: 13722")
SUMMARY["A0"] = a0.to_dict("records")

**✍️ Trả lời (sửa trực tiếp ô này):** Số đo có khớp bài báo không? Nếu lệch, nguyên nhân có thể là gì (dev set riêng, mẫu bị loại, phiên bản dữ liệu)?
> *Trả lời: ...*

In [ ]:
decide("A0", "Nguồn số liệu dùng trong báo cáo", "số đo được ở A0",
       "Bài báo có thể lệch phiên bản dữ liệu; báo cáo dùng số tự đo.")

### Lab A1. Quan hệ giữa GermanQuAD và GermanDPR
**Mục đích:** GermanDPR dẫn xuất từ GermanQuAD, nên có thể cùng câu hỏi/passage nằm ở cả hai bộ. DPR test (1.025) nhỏ hơn QuAD test (2.204): retrieval và end-to-end có thể đánh giá trên **hai tập câu hỏi khác nhau**.

In [ ]:
Q = {
  "QuAD.train": {norm(x["question"]) for x in qa["train"]},
  "QuAD.test":  {norm(x["question"]) for x in qa["test"]},
  "DPR.train":  {norm(r["q"]) for r in DPR["train"]},
  "DPR.test":   {norm(r["q"]) for r in DPR["test"]},
}
names = list(Q)
mat = pd.DataFrame([[len(Q[a] & Q[b]) for b in names] for a in names], index=names, columns=names)
display(mat)
print("Phần trăm câu hỏi DPR.test nằm trong QuAD.test:",
      round(100 * len(Q["DPR.test"] & Q["QuAD.test"]) / len(Q["DPR.test"]), 1), "%")
print("Phần trăm câu hỏi DPR.test nằm trong QuAD.train:",
      round(100 * len(Q["DPR.test"] & Q["QuAD.train"]) / len(Q["DPR.test"]), 1), "%")
save_table(mat.reset_index(), "A1_chong_lan_cau_hoi")
SUMMARY["A1"] = mat.to_dict()

In [ ]:
# Chồng lấn ở mức passage: positive của DPR có trùng context của QuAD không?
quad_ctx = {s: {pid(x["context"]) for x in qa[s]} for s in qa.keys()}
dpr_pos  = {s: {pid(r["pos"]) for r in DPR[s] if r["pos"]} for s in DPR}
for s_d in dpr_pos:
    for s_q in quad_ctx:
        inter = len(dpr_pos[s_d] & quad_ctx[s_q])
        print(f"DPR.{s_d} (positive duy nhất {len(dpr_pos[s_d])}) ∩ QuAD.{s_q} context: {inter}")

**✍️ Trả lời:** (1) DPR.test có phải tập con của QuAD.test không? (2) Vì sao điều này quan trọng khi bạn so sánh số đo retrieval với số đo end-to-end?
> *Trả lời: ...*

In [ ]:
decide("A1", "Tập câu hỏi dùng cho retrieval và end-to-end",
       "nêu rõ hai tập, hoặc dùng phần giao (sửa theo kết quả A1)",
       "DPR.test và QuAD.test có kích thước khác nhau nên số đo không so sánh trực tiếp được.")

### Lab A2. Chất lượng dữ liệu
**Mục đích:** tìm mẫu lỗi (span lệch, trùng lặp, trường rỗng) trước khi chúng làm sai số đo.

In [ ]:
def span_ok(x):
    texts, starts = qa_answers(x), qa_starts(x)
    return bool(texts) and all(x["context"][s:s+len(t)] == t for t, s in zip(texts, starts))

rows = []
for s in qa.keys():
    bad = [i for i, x in enumerate(qa[s]) if not span_ok(x)]
    qs  = [norm(x["question"]) for x in qa[s]]
    empty = sum(1 for x in qa[s] if not x["question"].strip() or not x["context"].strip() or not qa_answers(x))
    rows.append({"split": f"QuAD.{s}", "n": len(qa[s]), "span_lech": len(bad),
                 "cau_hoi_trung": len(qs) - len(set(qs)), "mau_rong": empty})
for s in DPR:
    qs = [norm(r["q"]) for r in DPR[s]]
    no_pos = sum(1 for r in DPR[s] if not r["pos"])
    no_ans_in_pos = sum(1 for r in DPR[s] if r["pos"] and not any(norm(a) in norm(r["pos"]) for a in r["ans"]))
    rows.append({"split": f"DPR.{s}", "n": len(DPR[s]), "span_lech": None,
                 "cau_hoi_trung": len(qs) - len(set(qs)), "mau_rong": no_pos,
                 "dap_an_khong_trong_positive": no_ans_in_pos})
a2 = save_table(pd.DataFrame(rows), "A2_chat_luong")
display(a2)
SUMMARY["A2"] = a2.to_dict("records")

**✍️ Trả lời:** Mẫu lỗi chiếm bao nhiêu phần trăm? Bạn loại hay giữ? Nếu `span_lech` > 0 do khác chuẩn hóa Unicode/khoảng trắng, đó là lỗi thật hay chỉ là khác biệt biểu diễn?
> *Trả lời: ...*

In [ ]:
decide("A2", "Xử lý mẫu lỗi", "giữ / loại (sửa)", "ghi số mẫu bị loại và lý do")

## Phần B. Phân tích thành phần bài toán (Tầng 2)
```
Câu hỏi ──► [Retriever] ──► top-k passage ──► [Reader] ──► span đáp án
             BM25 / dense / hybrid  (+ reranker)   extractive BERT/ELECTRA
```
| Thành phần | Cần gì từ dữ liệu | Rủi ro của bộ dữ liệu này |
|---|---|---|
| Corpus | passage duy nhất, quy mô vừa Colab | GermanDPR không có corpus riêng; phải tự gộp |
| Retriever | (câu hỏi, passage dương), âm khó | false negative; rò rỉ passage giữa split |
| Reader | (câu hỏi, context, span) | context vượt 512 token |
| Đánh giá | gold ở mức passage và mức đáp án | test có thể nhiều reference; chọn chỉ số chính |

**Bài tập B1 (✍️):** Sửa bảng trên bằng số đo thật của bạn sau khi chạy xong Phần C (thêm một cột "bằng chứng từ lab nào").

## Phần C. Từ yêu cầu đến phân tích (Tầng 3)
### Lab C1. Xây corpus và tập dev tạm
**Mục đích:** GermanDPR không có corpus riêng: gộp mọi positive + hard negative thành corpus, khử trùng lặp. Đồng thời tạo **tập dev theo nhóm passage** (xem lý do ở C7).

In [ ]:
corpus_text, corpus_id = [], {}          # corpus_id: pid -> chỉ số trong corpus_text
occ = Counter()
def add_passage(t):
    k = pid(t); occ[k] += 1
    if k not in corpus_id:
        corpus_id[k] = len(corpus_text); corpus_text.append(t)
    return corpus_id[k]

for s in DPR:
    for r in DPR[s]:
        r["pos_idx"]  = add_passage(r["pos"]) if r["pos"] else None
        r["hneg_idx"] = [add_passage(t) for t in r["hneg"]]

N = len(corpus_text); total_occ = sum(occ.values())
print(f"Tổng lượt xuất hiện: {total_occ} | passage duy nhất: {N} | tỉ lệ trùng: {1-N/total_occ:.1%}")

DIM = 384   # đổi nếu dùng mô hình khác
print(f"Ước lượng RAM chỉ mục flat float32 (dim={DIM}): {N*DIM*4/1e6:.0f} MB")
SUMMARY["C1"] = {"n_unique": N, "n_occurrences": total_occ}

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
train = DPR["train"]
groups = [r["pos_idx"] for r in train]            # nhóm = passage dương
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
tr_idx, dev_idx = next(gss.split(np.zeros(len(train)), groups=groups))
dev_idx = list(dev_idx); random.Random(42).shuffle(dev_idx)
DEV = [train[i] for i in dev_idx[:500]]            # 500 câu dev để lấy bằng chứng nhanh
print("train:", len(tr_idx), "| dev (nhóm):", len(dev_idx), "| dùng:", len(DEV))
print("Số câu hỏi trên mỗi passage dương (train):")
display(pd.Series(Counter(groups)).value_counts().sort_index().rename("so_passage").to_frame().T)

**✍️ Trả lời:** Corpus có bao nhiêu passage duy nhất, RAM chỉ mục bao nhiêu? Với số đó, FAISS flat có đủ không, hay cần IVF/HNSW? (Gợi ý: vài chục nghìn vector 384–768 chiều thì flat đủ nhanh.)
> *Trả lời: ...*

In [ ]:
decide("C1", "Phạm vi corpus", f"toàn bộ {N} passage duy nhất", "vừa RAM Colab; sửa nếu bạn quyết định lấy mẫu")
decide("C1", "Loại chỉ mục FAISS", "IndexFlatIP", "corpus nhỏ nên tìm chính xác, không cần xấp xỉ")

### Lab C2. Từ vựng tiếng Đức và tiền xử lý cho BM25
**Mục đích:** từ ghép dài và biến cách ảnh hưởng BM25. So sánh 3 cách xử lý trên 500 câu dev theo Recall@5 (khớp passage id).

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem.snowball import GermanStemmer
from functools import lru_cache
from rank_bm25 import BM25Okapi

stem = lru_cache(maxsize=None)(GermanStemmer().stem)
SW = set(stopwords.words("german"))
words = lambda s: re.findall(r"\w+", s.lower())
TOK = {
    "raw":       lambda s: words(s),
    "stem":      lambda s: [stem(w) for w in words(s)],
    "stem+stop": lambda s: [stem(w) for w in words(s) if w not in SW],
}

# thống kê từ ghép: tỉ lệ từ dài > 15 ký tự
allw = [w for t in corpus_text[:5000] for w in words(t)]
print("Tỉ lệ từ dài > 15 ký tự (ứng viên từ ghép):", round(np.mean([len(w) > 15 for w in allw]), 4))

In [ ]:
def topk_from_scores(scores, k):
    idx = np.argpartition(-scores, k)[:k]
    return idx[np.argsort(-scores[idx])]

def recall_at(rank_lists, gold, k=5):
    return float(np.mean([g in r[:k] for r, g in zip(rank_lists, gold)]))

gold = [r["pos_idx"] for r in DEV]
bm_models, res = {}, []
for name, tk in TOK.items():
    t0 = time.time()
    bm = BM25Okapi([tk(t) for t in corpus_text]); bm_models[name] = (bm, tk)
    ranks = [topk_from_scores(bm.get_scores(tk(r["q"])), 50) for r in DEV]
    res.append({"bien_the": name, "Recall@5": recall_at(ranks, gold, 5),
                "Recall@20": recall_at(ranks, gold, 20), "giay": round(time.time()-t0, 1)})
c2 = save_table(pd.DataFrame(res), "C2_bm25_tien_xu_ly")
display(c2)
BEST_BM = c2.sort_values("Recall@5", ascending=False).iloc[0]["bien_the"]
print("Biến thể BM25 tốt nhất trên dev:", BEST_BM)

**✍️ Trả lời:** Stemming/bỏ stopword giúp hay hại? Vì sao (gợi ý: biến cách, từ ghép không được tách bởi Snowball)?
> *Trả lời: ...*

In [ ]:
decide("C2", "Tiền xử lý BM25", BEST_BM, "tốt nhất theo Recall@5 trên 500 câu dev (không dùng test)")

### Lab C3. Độ dài passage và cắt cho dense retriever
**Mục đích:** bao nhiêu passage vượt 256/384/512 token? Đáp án có nằm sau điểm cắt không?

⚠️ Ngưỡng dài hơn chưa chắc tốt hơn: mô hình có thể được huấn luyện với độ dài ngắn hơn. Số liệu ở đây chỉ cho biết *có bị cắt mất thông tin không*, còn chọn ngưỡng cuối cùng phải kiểm bằng Recall (xem D1).

In [ ]:
from transformers import AutoTokenizer
DENSE_MODEL = "intfloat/multilingual-e5-small"     # có thể đổi; nhớ đổi DIM ở C1
tok_d = AutoTokenizer.from_pretrained(DENSE_MODEL)

sample_ids = random.Random(1).sample(range(N), min(3000, N))
enc = tok_d([corpus_text[i] for i in sample_ids], add_special_tokens=True,
            truncation=False, return_offsets_mapping=True)
lens = np.array([len(x) for x in enc["input_ids"]])
rows = [{"nguong": L, "ti_le_vuot": float((lens > L).mean())} for L in (128, 256, 384, 512)]
c3 = save_table(pd.DataFrame(rows), "C3_ti_le_vuot")
display(c3)
print("Phân vị độ dài (token): p50=%d p90=%d p99=%d max=%d" % tuple(np.percentile(lens, [50, 90, 99, 100])))
plt.hist(lens, bins=60); [plt.axvline(L, ls="--", c="r") for L in (256, 384, 512)]
plt.xlabel("số token / passage"); plt.ylabel("số passage"); save_fig("C3_do_dai_passage")

In [ ]:
# Đáp án có nằm sau điểm cắt L không? (chỉ xét positive, tìm vị trí ký tự của đáp án đầu tiên)
rows = []
pos_recs = [r for r in DPR["train"] if r["pos"]][:3000]
encp = tok_d([r["pos"] for r in pos_recs], truncation=False, return_offsets_mapping=True)
for L in (128, 256, 384, 512):
    cut_lost = tot = 0
    for r, offs in zip(pos_recs, encp["offset_mapping"]):
        p0 = r["pos"].find(r["ans"][0]) if r["ans"] else -1
        if p0 < 0: continue
        tot += 1
        if len(offs) > L and p0 >= offs[L-1][1]:      # đáp án bắt đầu sau ký tự cuối của token thứ L
            cut_lost += 1
    rows.append({"nguong": L, "dap_an_bi_cat_mat": cut_lost / max(tot, 1), "n_xet": tot})
c3b = save_table(pd.DataFrame(rows), "C3_dap_an_bi_cat")
display(c3b)

**✍️ Trả lời:** Ở ngưỡng 256 có bao nhiêu phần trăm đáp án bị cắt mất? Có nên dùng cửa sổ trượt cho passage dài, hay chấp nhận mất mát đó?
> *Trả lời: ...*

In [ ]:
MAXLEN = 256     # SỬA theo kết quả C3 (và kiểm lại bằng Recall ở D1)
decide("C3", "max_seq_length dense", MAXLEN, "dựa trên tỉ lệ vượt ngưỡng và tỉ lệ đáp án bị cắt; xác nhận bằng Recall ở D1")

### Lab C4. Đặc điểm đáp án và độ dài cho extractive reader
**Mục đích:** bao nhiêu mẫu cần cửa sổ trượt (`doc_stride`)? Đáp án dài bao nhiêu token? Đáp án thường nằm ở đâu?

In [ ]:
tok_r = AutoTokenizer.from_pretrained("deepset/gbert-base")
sub = random.Random(2).sample(range(len(qa["train"])), 3000)
xs = [qa["train"][i] for i in sub]
enc_qc = tok_r([x["question"] for x in xs], [x["context"] for x in xs], truncation=False)
len_qc = np.array([len(i) for i in enc_qc["input_ids"]])
len_ans = np.array([len(tok_r(qa_answers(x)[0], add_special_tokens=False)["input_ids"]) for x in xs])
rel_pos = np.array([qa_starts(x)[0] / max(len(x["context"]), 1) for x in xs])

print("Tỉ lệ (câu hỏi+context) > 512 token:", round(float((len_qc > 512).mean()), 4))
print("Độ dài context+câu hỏi p50/p95/p99:", np.percentile(len_qc, [50, 95, 99]).round())
print("Độ dài đáp án (token) p50/p95/p99/max:", np.percentile(len_ans, [50, 95, 99, 100]).round())
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].hist(len_qc, bins=50); ax[0].axvline(512, c="r", ls="--"); ax[0].set_title("câu hỏi+context (token)")
ax[1].hist(len_ans, bins=range(0, 60)); ax[1].set_title("độ dài đáp án (token)")
ax[2].hist(rel_pos, bins=20); ax[2].set_title("vị trí tương đối đáp án")
save_fig("C4_reader")
SUMMARY["C4"] = {"vuot_512": float((len_qc > 512).mean()), "ans_p99": float(np.percentile(len_ans, 99))}

**✍️ Trả lời:** Với tỉ lệ vượt 512 như trên, có bắt buộc dùng cửa sổ trượt không? Đáp án p99 dài bao nhiêu token và `max_answer_length` nên đặt ra sao?
> *Trả lời: ...*

In [ ]:
decide("C4", "max_seq_length reader / doc_stride", "384 / 128 (sửa)", "theo tỉ lệ context vượt ngưỡng ở C4")
decide("C4", "max_answer_length (token)", int(np.percentile(len_ans, 99)) + 5, "phân vị 99 độ dài đáp án cộng biên an toàn")

### Lab C5. Định nghĩa "truy hồi đúng" (false negative)
**Mục đích:** nếu passage khác (không phải positive) cũng chứa đáp án thì đánh giá theo *passage id* sẽ phạt oan retriever. Đo tỉ lệ đó trong hard negative.

In [ ]:
def has_ans(text, answers):
    t = norm(text)
    return any(a.strip() and norm(a) in t for a in answers)

fn_q, fn_examples = 0, []
for r in DEV:
    hits = [t for t in r["hneg"] if has_ans(t, r["ans"])]
    if hits:
        fn_q += 1
        if len(fn_examples) < 20: fn_examples.append({"cau_hoi": r["q"], "dap_an": r["ans"][0], "negative": hits[0][:300]})
print(f"Câu hỏi có ≥1 hard negative chứa đáp án: {fn_q}/{len(DEV)} = {fn_q/len(DEV):.1%}")
fn_df = save_table(pd.DataFrame(fn_examples), "C5_false_negative_vi_du")
display(fn_df.head(5))
SUMMARY["C5"] = {"ti_le_cau_co_FN": fn_q / len(DEV)}

**✍️ Trả lời:** Đọc tay 20 ví dụ trong `C5_false_negative_vi_du.csv`: bao nhiêu cái là *false negative thật* (passage đúng nghĩa trả lời được câu hỏi), bao nhiêu chỉ là trùng chuỗi ngẫu nhiên (ví dụ đáp án là số "1")?
> *Trả lời: ...*

In [ ]:
decide("C5", "Định nghĩa hit", "khớp passage id (sửa thành 'passage chứa đáp án' nếu FN đáng kể)",
       "dựa trên tỉ lệ false negative đo được và kết quả xem tay")

### Lab C6. EM / F1 và số reference
**Mục đích:** test có thể có nhiều đáp án tham chiếu cho một câu hỏi. Chỉ số chính phải chốt *trước khi xem kết quả mô hình*.

In [ ]:
for s in qa.keys():
    c = Counter(len(qa_answers(x)) for x in qa[s])
    print(f"QuAD.{s}: số reference/câu -> {dict(sorted(c.items()))}")

def norm_ans(s):
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\b(der|die|das|den|dem|des|ein|eine|einen|einem|einer|eines)\b", " ", s)
    return " ".join(s.split())

def em(pred, ref):  return float(norm_ans(pred) == norm_ans(ref))
def f1(pred, ref):
    p, r = norm_ans(pred).split(), norm_ans(ref).split()
    common = Counter(p) & Counter(r); same = sum(common.values())
    if not p or not r: return float(p == r)
    if same == 0: return 0.0
    pr, rc = same / len(p), same / len(r)
    return 2 * pr * rc / (pr + rc)

def score(pred, refs, mode="all_refs"):
    refs = refs if mode == "all_refs" else refs[:1]
    return max(em(pred, r) for r in refs), max(f1(pred, r) for r in refs)

# Thử trên dữ liệu thật: dùng reference thứ 2 làm "dự đoán" để thấy first_ref vs all_refs khác nhau thế nào
multi = [x for x in qa["test"] if len(qa_answers(x)) > 1]
if multi:
    em1 = np.mean([score(qa_answers(x)[1], qa_answers(x), "first_ref")[0] for x in multi])
    emA = np.mean([score(qa_answers(x)[1], qa_answers(x), "all_refs")[0] for x in multi])
    print(f"\nMinh họa: nếu dự đoán = reference #2 -> EM first_ref={em1:.3f}, EM all_refs={emA:.3f}")

**✍️ Trả lời:** Nếu test có nhiều reference, chỉ số `first_ref` và `all_refs` chênh nhau thế nào (xem minh họa)? Bạn chọn cái nào làm chỉ số **chính**, và cái còn lại báo cáo kèm?
> *Trả lời: ...*

In [ ]:
decide("C6", "Chỉ số QA chính", "first_ref (sửa) ; báo cáo kèm all_refs", "chốt trước khi xem kết quả mô hình để tránh chọn cho đẹp số")

### Lab C7. Chia dev và rò rỉ dữ liệu
**Mục đích:** nhiều câu hỏi có thể dùng chung một passage. Nếu chia dev theo *câu hỏi*, passage của dev rò sang train. Đo và so sánh với chia theo *nhóm passage*. Đo luôn mức "đã thấy" của test so với train.

In [ ]:
# 1) Chia ngẫu nhiên theo câu hỏi
rng = np.random.RandomState(42)
perm = rng.permutation(len(train)); cut = int(0.1 * len(train))
dev_q, tr_q = perm[:cut], perm[cut:]
train_pos_q = {train[i]["pos_idx"] for i in tr_q}
leak_q = np.mean([train[i]["pos_idx"] in train_pos_q for i in dev_q])

# 2) Chia theo nhóm passage (đã làm ở C1)
train_pos_g = {train[i]["pos_idx"] for i in tr_idx}
leak_g = np.mean([train[i]["pos_idx"] in train_pos_g for i in dev_idx])

# 3) Test: passage dương của DPR.test đã xuất hiện trong DPR.train (dương hoặc âm)?
train_all = {train[i]["pos_idx"] for i in range(len(train))} | {j for r in train for j in r["hneg_idx"]}
train_pos_all = {t["pos_idx"] for t in train}
test_pos_in_train_pos = np.mean([r["pos_idx"] in train_pos_all for r in DPR["test"]])
test_pos_in_train_any = np.mean([r["pos_idx"] in train_all for r in DPR["test"]])

c7 = save_table(pd.DataFrame([
    {"cach": "dev chia theo câu hỏi (passage dev đã có trong train)", "ti_le": leak_q},
    {"cach": "dev chia theo nhóm passage", "ti_le": leak_g},
    {"cach": "DPR.test: positive đã là positive của train", "ti_le": test_pos_in_train_pos},
    {"cach": "DPR.test: positive đã xuất hiện trong train (dương hoặc âm)", "ti_le": test_pos_in_train_any},
]), "C7_ro_ri")
display(c7)
SUMMARY["C7"] = c7.to_dict("records")

**✍️ Trả lời:** (1) Chia theo câu hỏi rò bao nhiêu? (2) Nếu retriever được fine-tune trên DPR.train rồi đánh giá trên passage đã thấy, điều gì xảy ra với số đo? (3) Vì sao cần một tập "unseen" (passage chưa thấy) để chọn cấu hình?
> *Trả lời: ...*

In [ ]:
decide("C7", "Cách chia dev", "theo nhóm passage (GroupShuffleSplit)", f"chia theo câu hỏi rò {leak_q:.1%} passage, theo nhóm rò {leak_g:.1%}")
decide("C7", "Tập chọn cấu hình", "dev-unseen (passage chưa xuất hiện ở train)", "tránh số đo đẹp giả do passage đã thấy; test chỉ để báo cáo")

### Lab C9. Thiết kế error decomposition
**Mục đích:** định nghĩa *trước khi xem kết quả* cách phân loại lỗi end-to-end. Ở đây chỉ viết và kiểm thử hàm gán nhãn trên dữ liệu giả lập.

In [ ]:
K_SUCCESS = 5      # ngưỡng "retrieval thành công": passage đúng nằm trong top-K

def classify(gold_idx, retrieved_idx, pred_ans, refs):
    """success | retrieval_miss | reader_miss"""
    if gold_idx not in retrieved_idx[:K_SUCCESS]:
        return "retrieval_miss"
    ok = max(em(pred_ans, r) for r in refs)
    return "success" if ok == 1.0 else "reader_miss"

# Kiểm thử bằng 3 tình huống giả lập
tests = [
    (7, [1, 2, 3, 4, 5, 6], "Berlin", ["Berlin"], "retrieval_miss"),
    (3, [9, 3, 1, 2, 4, 5], "Bonn",   ["Berlin"], "reader_miss"),
    (3, [9, 3, 1, 2, 4, 5], "Berlin", ["Berlin"], "success"),
]
for g, ret, pred, refs, want in tests:
    got = classify(g, ret, pred, refs)
    print(("OK  " if got == want else "LỖI"), got, "(mong đợi:", want, ")")

**✍️ Trả lời:** Vì sao phải chốt `K_SUCCESS` trước khi xem số liệu? Nếu retrieval đúng ở hạng 20 nhưng bạn chỉ đưa top-5 vào reader, mẫu đó thuộc loại lỗi nào và vì sao phân biệt này quan trọng cho việc cải tiến?
> *Trả lời: ...*

In [ ]:
decide("C9", "K ngưỡng retrieval thành công", K_SUCCESS, "bằng số passage đưa vào reader; chốt trước khi xem kết quả")

## Phần D. Bằng chứng cho các cải tiến (Tầng 4)
> Chỉ dùng **500 câu dev**. Đây là thí nghiệm nhỏ để lấy bằng chứng, **không phải** đánh giá chính thức. Không dùng test.

### Lab D1. Encode dense, so sánh BM25 và dense, thử hybrid
⚠️ **Chọn GPU T4 trước khi chạy ô encode** (≈ vài phút với e5-small; trên CPU sẽ rất chậm).

In [ ]:
import torch, faiss
from sentence_transformers import SentenceTransformer
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
model = SentenceTransformer(DENSE_MODEL, device=device)
model.max_seq_length = MAXLEN
emb_path = f"{OUT}/emb_{DENSE_MODEL.split('/')[-1]}_{MAXLEN}.npy"
if os.path.exists(emb_path):
    EMB = np.load(emb_path); print("Nạp embedding đã lưu:", EMB.shape)
else:
    t0 = time.time()
    EMB = model.encode(["passage: " + t for t in corpus_text], batch_size=64,
                       normalize_embeddings=True, show_progress_bar=True).astype("float32")
    np.save(emb_path, EMB); print(f"Encode {len(EMB)} passage trong {time.time()-t0:.0f}s")
index = faiss.IndexFlatIP(EMB.shape[1]); index.add(EMB)

In [ ]:
def dense_rank(qs, k=50):
    qv = model.encode(["query: " + q for q in qs], batch_size=64, normalize_embeddings=True).astype("float32")
    return index.search(qv, k)[1]

# BM25 (biến thể tốt nhất ở C2) -> top-50 và điểm của positive / hard negative (dùng lại ở D2, D3)
bm, tk = bm_models[BEST_BM]
bm_rank, bm_pos_score, bm_neg_scores = [], [], []
for r in DEV:
    s = bm.get_scores(tk(r["q"]))
    bm_rank.append(topk_from_scores(s, 50))
    bm_pos_score.append(s[r["pos_idx"]])
    bm_neg_scores.append([s[j] for j in r["hneg_idx"]])
de_rank = dense_rank([r["q"] for r in DEV])

def rrf(lists, k=60, top=50):
    sc = defaultdict(float)
    for lst in lists:
        for rank, d in enumerate(lst): sc[d] += 1 / (k + rank + 1)
    return [d for d, _ in sorted(sc.items(), key=lambda x: -x[1])[:top]]
hy_rank = [rrf([b, d]) for b, d in zip(bm_rank, de_rank)]

res = [{"he_thong": n, **{f"Recall@{k}": recall_at(rl, gold, k) for k in (1, 5, 20)}}
       for n, rl in (("BM25", bm_rank), ("Dense", de_rank), ("Hybrid RRF", hy_rank))]
d1 = save_table(pd.DataFrame(res), "D1_retrieval_dev500"); display(d1)

In [ ]:
# Bảng 2x2 trên top-5: ai đúng, ai sai
b_ok = np.array([g in r[:5] for r, g in zip(bm_rank, gold)])
d_ok = np.array([g in r[:5] for r, g in zip(de_rank, gold)])
tab = pd.DataFrame({"BM25 đúng": [int((b_ok & d_ok).sum()), int((b_ok & ~d_ok).sum())],
                    "BM25 sai":  [int((~b_ok & d_ok).sum()), int((~b_ok & ~d_ok).sum())]},
                   index=["Dense đúng", "Dense sai"])
display(tab)
union = float((b_ok | d_ok).mean())
print(f"Trần hợp nhất (ít nhất một bên đúng, top-5): {union:.3f} | tốt nhất đơn lẻ: {max(b_ok.mean(), d_ok.mean()):.3f}")
SUMMARY["D1"] = {"union_top5": union, "bm25": float(b_ok.mean()), "dense": float(d_ok.mean())}

**✍️ Trả lời:** Hai nhóm "chỉ một bên đúng" có đáng kể không? Trần hợp nhất cao hơn tốt nhất đơn lẻ bao nhiêu điểm phần trăm? Hybrid RRF có vượt cả hai không? Nếu dense thấp hơn BM25, hãy nghĩ đến: mô hình chưa fine-tune cho tiếng Đức, hay passage đã bị cắt (so MAXLEN với C3)?
> *Trả lời: ...*

In [ ]:
decide("D1", "Làm hybrid BM25 + dense?", "có / không (sửa)", f"trần hợp nhất {union:.3f} vs tốt nhất đơn lẻ {max(b_ok.mean(), d_ok.mean()):.3f}")

### Lab C8. Ước lượng latency từng bước
**Mục đích:** biết bước nào đắt để chọn `k` và quyết định có rerank hay không. Mỗi phép đo chạy nhiều lần, báo cáo trung vị và p95.

In [ ]:
def bench(fn, n=100):
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1000)
    return np.median(ts), np.percentile(ts, 95)

qs = [r["q"] for r in DEV[:100]]; it = iter(range(10**9))
def f_bm25():  bm.get_scores(tk(qs[next(it) % 100]))
def f_enc():   model.encode(["query: " + qs[next(it) % 100]], normalize_embeddings=True)
qv0 = model.encode(["query: " + qs[0]], normalize_embeddings=True).astype("float32")
def f_faiss(): index.search(qv0, 50)

rows = [{"buoc": n, "p50_ms": round(a, 2), "p95_ms": round(b_, 2)}
        for n, (a, b_) in (("BM25 get_scores", bench(f_bm25, 30)),
                           ("Encode 1 câu hỏi", bench(f_enc)),
                           ("FAISS search top-50", bench(f_faiss)))]
c8 = save_table(pd.DataFrame(rows), "C8_latency"); display(c8)
print("Gợi ý: đo thêm reader trên 1 passage (cài ở giai đoạn sau) và nhân với k để biết chi phí end-to-end.")

In [ ]:
decide("C8", "Số passage đưa vào reader (k)", 5, "chi phí reader tăng tuyến tính theo k; sửa theo số đo latency và C9")

### Lab D2. Khi retriever sai ở top-5, passage đúng nằm ở hạng nào?
**Mục đích:** nếu passage đúng thường ở hạng 6–50 thì reranker có thể cứu được; nếu nằm ngoài top-50 thì rerank vô ích và phải cải thiện retriever.

In [ ]:
def gold_rank(rl, g):
    return rl.index(g) + 1 if g in rl else None

rows = []
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for a, (name, rls) in zip(ax, (("BM25", bm_rank), ("Dense", de_rank), ("Hybrid", hy_rank))):
    rk = [gold_rank(list(map(int, r)), g) for r, g in zip(rls, gold)]
    miss5 = [x for x in rk if x is None or x > 5]
    in6_50 = [x for x in miss5 if x is not None]
    rows.append({"he_thong": name, "sai_top5": len(miss5), "co_trong_top50": len(in6_50),
                 "ngoai_top50": len(miss5) - len(in6_50),
                 "ti_le_cuu_duoc_neu_rerank": len(in6_50) / max(len(miss5), 1)})
    a.hist(in6_50, bins=range(6, 52, 3)); a.set_title(f"{name}: hạng passage đúng (khi sai top-5)")
save_fig("D2_hang_passage_dung")
d2 = save_table(pd.DataFrame(rows), "D2_cuu_duoc_neu_rerank"); display(d2)

**✍️ Trả lời:** Với hybrid, bao nhiêu phần trăm câu sai top-5 có passage đúng trong top-50? Nên đưa bao nhiêu ứng viên vào reranker (top-20 hay top-50)? Chi phí rerank tăng tuyến tính theo số ứng viên, hãy đối chiếu với latency ở C8.
> *Trả lời: ...*

In [ ]:
decide("D2", "Số ứng viên đưa vào reranker", 20, "sửa theo tỉ lệ cứu được ở D2 và chi phí latency ở C8")

### Lab D3. Chất lượng hard negative
**Mục đích:** hard negative có sẵn có thật sự "khó" không? Có false negative không? Nếu negative điểm cao hơn positive thì có thể là nhiễu nhãn.

In [ ]:
neg_gt_pos, n_neg_total, n_q_with = 0, 0, 0
for ps, ns in zip(bm_pos_score, bm_neg_scores):
    if not ns: continue
    c = sum(n > ps for n in ns); neg_gt_pos += c; n_neg_total += len(ns); n_q_with += (c > 0)
print(f"BM25: negative có điểm cao hơn positive: {neg_gt_pos}/{n_neg_total} = {neg_gt_pos/max(n_neg_total,1):.1%}")
print(f"Câu hỏi có ≥1 negative điểm cao hơn positive: {n_q_with}/{len(DEV)} = {n_q_with/len(DEV):.1%}")

allpos = np.array(bm_pos_score); allneg = np.array([x for ns in bm_neg_scores for x in ns])
plt.hist(allpos, bins=40, alpha=.6, label="positive"); plt.hist(allneg, bins=40, alpha=.6, label="hard negative")
plt.legend(); plt.xlabel("điểm BM25"); save_fig("D3_diem_positive_vs_negative")

# 20 negative điểm cao nhất kèm cờ "chứa đáp án" để xem tay
cands = []
for r, ps, ns in zip(DEV, bm_pos_score, bm_neg_scores):
    for t, s in zip(r["hneg"], ns):
        cands.append({"q": r["q"], "dap_an": r["ans"][0] if r["ans"] else "", "diem_neg": s,
                      "diem_pos": ps, "chua_dap_an": has_ans(t, r["ans"]), "negative": t[:300]})
top20 = pd.DataFrame(cands).sort_values("diem_neg", ascending=False).head(20)
save_table(top20, "D3_negative_diem_cao_xem_tay"); display(top20[["q", "diem_neg", "diem_pos", "chua_dap_an"]].head(8))

**✍️ Trả lời:** Đọc tay `D3_negative_diem_cao_xem_tay.csv`: bao nhiêu trong 20 cái thật sự là false negative? Hard negative sẵn có có đủ khó để huấn luyện không, hay nên khai thác lại từ chính retriever?
> *Trả lời: ...*

In [ ]:
decide("D3", "Chiến lược hard negative", "dùng sẵn có + lọc FN (sửa)", f"{n_q_with/len(DEV):.1%} câu có negative điểm cao hơn positive; xác nhận bằng xem tay")

### Lab D4. Nhóm để phân tích robustness
**Mục đích:** chỉ so sánh các nhóm có đủ mẫu (gợi ý ≥ 50). Đếm trước nhóm nào đủ, và nhìn sơ Recall@5 theo nhóm.

In [ ]:
WH = ["wer", "was", "wann", "wo", "wie viele", "wie", "warum", "welche", "wie viel"]
def qtype(q):
    ql = q.lower()
    for w in sorted(WH, key=len, reverse=True):
        if re.search(rf"\b{w}\b", ql): return w
    return "khác"

train_q = [r["q"] for r in train]
cnt = Counter(qtype(q) for q in train_q)
print("Loại câu hỏi (toàn bộ DPR.train):", dict(cnt.most_common()))

df = pd.DataFrame({"qtype": [qtype(r["q"]) for r in DEV],
                   "q_len": [len(words(r["q"])) for r in DEV],
                   "p_len": [len(words(r["pos"])) for r in DEV],
                   "bm25": b_ok, "dense": d_ok})
df["q_len_nhom"] = pd.qcut(df["q_len"], 3, labels=["ngan", "vua", "dai"], duplicates="drop")
df["p_len_nhom"] = pd.qcut(df["p_len"], 3, labels=["ngan", "vua", "dai"], duplicates="drop")
for col in ("qtype", "q_len_nhom", "p_len_nhom"):
    g = df.groupby(col, observed=True).agg(n=("bm25", "size"), bm25_R5=("bm25", "mean"), dense_R5=("dense", "mean")).round(3)
    g["du_mau(>=50)"] = g["n"] >= 50
    display(save_table(g.reset_index(), f"D4_{col}"))

**✍️ Trả lời:** Nhóm nào đủ mẫu để phân tích robustness? Có nhóm nào BM25 và dense chênh nhau rõ (gợi ý cho hybrid)? Lưu ý: với n nhỏ, chênh lệch có thể chỉ là nhiễu, hãy nói rõ trong báo cáo.
> *Trả lời: ...*

In [ ]:
big = df["qtype"].value_counts()
decide("D4", "Nhóm phân tích robustness", list(big[big >= 50].index), "chỉ các loại câu hỏi có ≥ 50 mẫu trong dev 500")

## Phần E. Tổng hợp
Bảng *Quyết định* là cầu nối giữa EDA và cài đặt. Với mỗi dòng, bạn cần thêm cột **Phát hiện (kèm số liệu)** và **Phần cài đặt bị ảnh hưởng** khi viết báo cáo.

In [ ]:
final = pd.DataFrame(DECISIONS)
final["phat_hien_kem_so_lieu"] = ""          # ✍️ điền: số đo hỗ trợ quyết định
final["phan_cai_dat_anh_huong"] = ""         # ✍️ điền: BM25 / dense / reader / đánh giá ...
display(final)
save_table(final, "E_bang_phat_hien_quyet_dinh")

with open(f"{OUT}/eda_summary.json", "w", encoding="utf-8") as f:
    json.dump({"summary": SUMMARY, "decisions": DECISIONS}, f, ensure_ascii=False, indent=2, default=str)
print("Đã lưu eda_summary.json và các bảng/hình tại", OUT)

### Việc cần làm cuối
1. Mở `E_bang_phat_hien_quyet_dinh.csv`, điền hai cột còn trống.
2. Kiểm tra mọi ô ✍️ đã có câu trả lời của riêng bạn.
3. Chạy lại **Runtime > Restart and run all** để chắc notebook tái lập được.
4. Nộp: notebook + thư mục kết quả + báo cáo 4–6 trang.

**Nhắc lại các bẫy:** chia dev theo nhóm passage; không dùng test để chọn thiết kế; chốt chỉ số chính và `K_SUCCESS` trước khi xem kết quả; số trong bài báo không thay thế số bạn đo được.